In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/email_feature_engineered.csv')

In [12]:
feature_inventory = pd.DataFrame({
    'Feature': df.columns,
    'Data Type': df.dtypes.astype(str).values,
    'Missing Values': df.isnull().sum().values,
    'Missing Percentage': (
        df.isnull().mean().values * 100
    ).round(2),
    'Unique Values': df.nunique().values
})

display(
    feature_inventory.sort_values(
        by='Unique Values',
        ascending=False
    )
)

,Feature,Data Type,Missing Values,Missing Percentage,Unique Values
4,date,str,432,0.4,106835
7,body,str,0,0.0,103518
5,subject,str,0,0.0,68486
0,sender,str,0,0.0,55562
8,urls,str,0,0.0,38353
1,sender_domain,str,0,0.0,28465
2,receiver,str,0,0.0,17581
3,receiver_domain,str,0,0.0,6367
11,url_length_avg,float64,0,0.0,4497
22,url_length_avg_log,float64,0,0.0,4497


In [13]:
# Columns to remove
removed_features = [
    'sender',
    'receiver',
    'date',
    'source',
    'url_length_max',
    'url_length_avg',
    'url_subdom_max',
    'url_subdom_avg',
    'attachment_count',
    'has_attachments',
    'has_attachments_engineered',
    'has_url',
    'has_multiple_receiver_domains',
    'has_html_content',
    'has_plain_text',
    'has_mixed_content',
    'language_count'
]

# Drop columns
df.drop(
    columns=removed_features,
    inplace=True
)

print("Updated Dataset Shape:", df.shape)

display(df.head())

Updated Dataset Shape: (108684, 18)


C:\Users\dhara\AppData\Local\Temp\ipykernel_26740\604664354.py:23: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(


,sender_domain,receiver_domain,subject,content_types,body,urls,url_count,attachment_types,language,label,url_length_max_log,url_length_avg_log,url_subdom_max_log,url_subdom_avg_log,attachment_count_log,receiver_domain_count,receiver_unique_domain_count,date_year
0,enron.com,enron.com,[ORGANIZATION] failover plan.,text/plain,"Hi [NAME], \n\nTonight we are rolling out a n...",[NO URL],0.0,none,en,0.0,0.000000,0.000000,0.000000,0.000000,0.0,1,1,2001.0
1,enron.com,enron.com,RE: Intranet Site,text/plain,"[NAME] r these new?\tIntranet Site\n\n[NAME],\...",http://eastpower.dev.corp.enron.com/summary/pj...,3.0,none,en,0.0,4.110874,4.077537,1.386294,1.386294,0.0,1,1,2001.0
2,enron.com,enron.com;enron.com,FW: [ORGANIZATION] Company information,text/plain,"[NAME]/[NAME],\n\nWe are currently trading und...",[NO URL],0.0,none,en,0.0,0.000000,0.000000,0.000000,0.000000,0.0,2,1,2001.0
3,enron.com,enron.com;enron.com,New Master Physical,text/plain,[NAME] and [NAME] -\n\nAttached is a worksheet...,[NO URL],0.0,none,en,0.0,0.000000,0.000000,0.000000,0.000000,0.0,2,1,2001.0
4,enron.com,enron.com;enron.com;enron.com,FW: [ORGANIZATION]/Mirant GISB,text/plain,FYI. Below is a copy of my communication with ...,[NO URL],0.0,none,en,0.0,0.000000,0.000000,0.000000,0.000000,0.0,3,1,2001.0


In [15]:
# Handling missing values in the date_year column
print("Missing values:", df['date_year'].isna().sum())

print("\nYear distribution:")
print(df['date_year'].describe())

Missing values: 432

Year distribution:
count    108252.000000
mean       2003.514725
std           3.121076
min        1990.000000
25%        2001.000000
50%        2002.000000
75%        2007.000000
max        2025.000000
Name: date_year, dtype: float64


In [16]:
date_year_median = df['date_year'].median()

print("Median Year:", date_year_median)

Median Year: 2002.0


In [18]:
df['date_missing'] = df['date_year'].isna().astype('int8')

print(df['date_missing'].value_counts())

date_missing
0    108252
1       432
Name: count, dtype: int64


In [19]:
print("Missing date_year:", df['date_year'].isna().sum())

print("Missing date_missing:", df['date_missing'].isna().sum())

print("\nDate missing distribution:")
print(df['date_missing'].value_counts())

Missing date_year: 432
Missing date_missing: 0

Date missing distribution:
date_missing
0    108252
1       432
Name: count, dtype: int64


#### Step 3: Feature Encoding & Preprocessing

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)

numeric_features = [
    'url_count',
    'url_length_max_log',
    'url_length_avg_log',
    'url_subdom_max_log',
    'url_subdom_avg_log',
    'attachment_count_log',
    'receiver_domain_count',
    'receiver_unique_domain_count',
    'date_year',
    'date_missing'
]

categorical_features = [
    'sender_domain',
    'content_types',
    'attachment_types'
]

multilabel_features = [
    'receiver_domain',
    'language'
]

text_features = [
    'subject',
    'body',
    'urls'
]

TARGET = 'label'

In [21]:
df.loc[
    df['date_missing'] == 1,
    'date_year'
] = np.nan

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (86947, 18)
Testing: (21737, 18)


In [23]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

In [24]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='UNKNOWN'
    )),
    ('encoder', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=20
    ))
])

In [25]:
def semicolon_tokenizer(value):
    return [
        item.strip().lower()
        for item in value.split(';')
        if item.strip()
    ]

In [26]:
receiver_vectorizer = CountVectorizer(
    tokenizer=semicolon_tokenizer,
    token_pattern=None,
    binary=True,
    min_df=5,
    max_features=5000
)

In [27]:
language_vectorizer = CountVectorizer(
    tokenizer=semicolon_tokenizer,
    token_pattern=None,
    binary=True,
    min_df=2
)

In [28]:
subject_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

In [29]:
body_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000,
    sublinear_tf=True
)

In [30]:
url_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    min_df=3,
    max_features=20000
)

In [31]:
# Combine everything using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', numeric_pipeline, numeric_features),

        ('categorical', categorical_pipeline, categorical_features),

        ('receiver_domains', receiver_vectorizer, 'receiver_domain'),

        ('languages', language_vectorizer, 'language'),

        ('subject_tfidf', subject_vectorizer, 'subject'),

        ('body_tfidf', body_vectorizer, 'body'),

        ('url_tfidf', url_vectorizer, 'urls')
    ],
    remainder='drop'
)

In [32]:
for col in multilabel_features + text_features:
    X_train[col] = X_train[col].fillna('').astype(str)
    X_test[col] = X_test[col].fillna('').astype(str)

In [33]:
# Fit and transform the training data
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)

In [34]:
print("Training Matrix:", X_train_encoded.shape)

print("Testing Matrix:", X_test_encoded.shape)

print("Training Matrix Type:", type(X_train_encoded))

print("Training Matrix Format:", X_train_encoded.format)

Training Matrix: (86947, 81301)
Testing Matrix: (21737, 81301)
Training Matrix Type: <class 'scipy.sparse._csr.csr_matrix'>
Training Matrix Format: csr


In [35]:
memory_bytes = (
    X_train_encoded.data.nbytes +
    X_train_encoded.indices.nbytes +
    X_train_encoded.indptr.nbytes
)

print(
    f"Memory Usage: {memory_bytes / (1024**2):.2f} MB"
)

Memory Usage: 276.66 MB


In [36]:
from scipy.sparse import save_npz
from scipy.sparse import load_npz
from pathlib import Path

ARTIFACT_DIR = Path("../models/feature_selection")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save sparse matrices
save_npz(
    ARTIFACT_DIR / "encoded_features.npz",
    X_train_encoded
)

save_npz(
    ARTIFACT_DIR / "encoded_test_features.npz",
    X_test_encoded
)

In [37]:
import pandas as pd

pd.DataFrame({
    'y_train': y_train.reset_index(drop=True)
}).to_csv(ARTIFACT_DIR / "y_train.csv", index=False)

pd.DataFrame({
    'y_test': y_test.reset_index(drop=True)
}).to_csv(ARTIFACT_DIR / "y_test.csv", index=False)

In [38]:
import joblib

joblib.dump(
    preprocessor,
    ARTIFACT_DIR / "preprocessor.joblib",
    compress=3
)

['..\\models\\feature_selection\\preprocessor.joblib']